## 🧩 Workshop Structure (90 Minutes)

**Group**: 1  
**Team Members:**  Zhuoran Zhang(9048508),  Ce Chen(9007166)

### Part A — Build the Corpus Download a large corpus and create a document collection.
 You must report:  
- corpus source  
- number of documents   
- approximate vocabulary size   
- a short description of the domain  

In [1]:
# ✅ Download FULL blog posts directly into 'sample_docs'

import os
import requests
import feedparser
from bs4 import BeautifulSoup

def fetch_full_blog_posts(rss_url, save_folder="sample_docs", max_posts=200):
    feed = feedparser.parse(rss_url)

    # Create folder if it doesn't exist
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)

    count = 0

    for entry in feed.entries:
        if count >= max_posts:
            break

        title = entry.get("title", f"untitled_{count+1}")
        link = entry.get("link", "")
        summary = entry.get("summary", "")

        content = ""

        try:
            response = requests.get(link, timeout=15)
            soup = BeautifulSoup(response.text, "html.parser")

            # Remove unnecessary tags
            for tag in soup(["script", "style", "noscript"]):
                tag.extract()

            # Try to extract main content
            section = (
                soup.select_one(".post-body") or
                soup.select_one(".entry-content") or
                soup.select_one("article")
            )

            if section:
                content = section.get_text(separator="\n", strip=True)
            else:
                # Fallback: extract all paragraphs
                paragraphs = soup.find_all("p")
                content = "\n".join(p.get_text(strip=True) for p in paragraphs)

        except Exception as e:
            print(f"Warning: Failed to fetch full content for {title}. Using summary.")
            content = BeautifulSoup(summary, "html.parser").get_text()

        # Save file
        filename = f"blog_{count+1}.txt"
        filepath = os.path.join(save_folder, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"{title}\n\n{content}")

        count += 1

    print(f"Saved {count} full blog posts to '{save_folder}'")


def load_documents(folder_path="sample_docs"):
    documents = []
    filenames = []

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, "r", encoding="utf-8") as file:
                documents.append(file.read())
                filenames.append(filename)

    return filenames, documents


# Recommended higher-quality source
rss_url = "https://realpython.com/atom.xml"

# Step 1: download and save files
fetch_full_blog_posts(rss_url, "sample_docs", max_posts=200)

# Step 2: load files back into Python
filenames, documents = load_documents("sample_docs")

print(f"Loaded {len(documents)} documents.")
print("First file:", filenames[0])
print("\nPreview:\n")
print(documents[0][:500])

Saved 40 full blog posts to 'sample_docs'
Loaded 40 documents.
First file: blog_1.txt

Preview:

The Real Python Podcast – Episode #289: Limitations in Human and Automated Code Review

With the mountains of Python code that it’s possible to generate now, how’s your code review going? What are the limitations of human review, and where does machine review excel? Christopher Trudeau is back on the show this week with another batch of PyCoder’s Weekly articles and projects.
Episode Sponsor:
We discuss a recent piece from Glyph titled, “What Is Code Review For?” We dig into the limitations of h


### Part B — Build the Retrieval Pipeline

Implement the following:

1. **Tokenizer**
2. **Normalization**
3. **Stop-word removal**
4. **Stemming or lemmatization**
5. **Term-document incidence matrix**
6. **Term frequency**
7. **Log-frequency weighting**
8. **Document frequency**
9. **Inverse document frequency**
10. **TF-IDF weighting**
11. **Cosine similarity retrieval**

### 🔹 Tokenization

Tokenization is the process of breaking text into smaller units called tokens, typically words.

This step converts raw text into a list of words that can be processed computationally.

For example:

"Machine learning is powerful!" → ["machine", "learning", "is", "powerful"]

Tokenization is the first step in any Information Retrieval (IR) pipeline, as it transforms unstructured text into structured data.

In [2]:
# Tokenizer

import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    return tokens

# Example
sample_tokens = tokenize(documents[0])
print(sample_tokens[:20])
print(f"Total tokens: {len(sample_tokens)}")

['the', 'real', 'python', 'podcast', 'episode', 'limitations', 'in', 'human', 'and', 'automated', 'code', 'review', 'with', 'the', 'mountains', 'of', 'python', 'code', 'that', 'it']
Total tokens: 231


### 🔹 Normalization

Normalization standardizes text to ensure consistency across documents.

Common normalization steps include:

- Converting text to lowercase
- Removing punctuation and special characters
- Cleaning formatting inconsistencies

For example:

"Python, PYTHON!" → "python python"

In this project, normalization is applied during tokenization by converting text to lowercase and removing non-alphanumeric characters.

In [3]:
# Normalization

def normalize(tokens):
    normalized_tokens = [token.lower() for token in tokens]
    return normalized_tokens

# Example
normalized_tokens = normalize(sample_tokens)
print(normalized_tokens[:20])

['the', 'real', 'python', 'podcast', 'episode', 'limitations', 'in', 'human', 'and', 'automated', 'code', 'review', 'with', 'the', 'mountains', 'of', 'python', 'code', 'that', 'it']


### 🔹 Stop-word Removal

Stop-word removal eliminates common words that carry little semantic meaning, such as:

"the", "is", "and", "of"

These words appear frequently but do not help distinguish between documents.

Removing stop-words:

- Reduces noise
- Improves efficiency
- Enhances retrieval quality

For example:

"This is a simple example" → ["simple", "example"]

In [4]:
# Stopword Removal

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    filtered_tokens = [token for token in tokens if token not in stop_words]
    return filtered_tokens

# Example
filtered_tokens = remove_stopwords(normalized_tokens)
print(filtered_tokens[:20])

['real', 'python', 'podcast', 'episode', 'limitations', 'human', 'automated', 'code', 'review', 'mountains', 'python', 'code', 'possible', 'generate', 'code', 'review', 'going', 'limitations', 'human', 'review']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\85155\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 🔹 Stemming / Lemmatization

Stemming and lemmatization reduce words to their base or root form.

- Stemming: applies simple rules (may produce non-words)
  - "running" → "run"
  - "studies" → "studi"

- Lemmatization: uses linguistic knowledge (more accurate)
  - "running" → "run"
  - "better" → "good"

In this project, stemming is used to reduce vocabulary size and group similar terms.

In [5]:
# Stemming

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_tokens(tokens):
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return stemmed_tokens

# Example
stemmed_tokens = stem_tokens(filtered_tokens)
print(stemmed_tokens[:20])

['real', 'python', 'podcast', 'episod', 'limit', 'human', 'autom', 'code', 'review', 'mountain', 'python', 'code', 'possibl', 'gener', 'code', 'review', 'go', 'limit', 'human', 'review']


### 🔹 Term-Document Incidence Matrix

A Term-Document Incidence Matrix is a binary representation of whether a term appears in a document.

- Rows represent terms
- Columns represent documents
- Values are:
  - 1 → term appears in the document
  - 0 → term does not appear

This matrix provides a simple way to represent document content and is often used in basic retrieval models.

Example:

| Term   | Doc1 | Doc2 |
|--------|------|------|
| python | 1    | 0    |
| data   | 1    | 1    |

In [6]:
# Term-document incidence matrix

from collections import defaultdict
import pandas as pd

def preprocess_text(text):
    tokens = tokenize(text)
    tokens = normalize(tokens)
    tokens = remove_stopwords(tokens)
    tokens = stem_tokens(tokens)   # or use lemmatize_tokens(tokens)
    return tokens

def build_incidence_matrix(documents):
    term_doc = defaultdict(set)

    for doc_id, text in enumerate(documents):
        tokens = preprocess_text(text)
        for token in set(tokens):
            term_doc[token].add(doc_id)

    terms = sorted(term_doc.keys())
    matrix = []

    for term in terms:
        row = []
        for doc_id in range(len(documents)):
            row.append(1 if doc_id in term_doc[term] else 0)
        matrix.append(row)

    incidence_df = pd.DataFrame(
        matrix,
        index=terms,
        columns=[f"Doc_{i}" for i in range(len(documents))]
    )
    return incidence_df

incidence_matrix = build_incidence_matrix(documents)
incidence_matrix.head(10)

,Doc_0,Doc_1,Doc_2,Doc_3,Doc_4,Doc_5,Doc_6,Doc_7,Doc_8,Doc_9,...,Doc_30,Doc_31,Doc_32,Doc_33,Doc_34,Doc_35,Doc_36,Doc_37,Doc_38,Doc_39
aarch,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
abil,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
abl,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0
aboutasynchron,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
aboutchristoph,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
aboutdaria,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
abouterik,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
aboutforloop,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
abouthellen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
aboutisabel,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 🔹 Term Frequency (TF)

Term Frequency measures how often a term appears in a document.

It is usually normalized by document length:

TF(t, d) = (number of occurrences of t in d) / (total terms in d)

TF captures the importance of a term within a specific document.

Higher TF means the term is more representative of that document.

In [7]:
# Term frequency

from collections import Counter

def build_term_counts(documents):
    doc_term_counts = []
    doc_lengths = []

    for doc in documents:
        tokens = preprocess_text(doc)
        counts = Counter(tokens)
        doc_term_counts.append(counts)
        doc_lengths.append(sum(counts.values()))

    return doc_term_counts, doc_lengths

def compute_tf(doc_term_counts, doc_lengths):
    tf = []

    for i, counts in enumerate(doc_term_counts):
        doc_tf = {}
        for term, count in counts.items():
            doc_tf[term] = count / doc_lengths[i]
        tf.append(doc_tf)

    return tf

doc_term_counts, doc_lengths = build_term_counts(documents)
tf = compute_tf(doc_term_counts, doc_lengths)

print(dict(list(tf[0].items())[:10]))

{'real': 0.006578947368421052, 'python': 0.06578947368421052, 'podcast': 0.006578947368421052, 'episod': 0.019736842105263157, 'limit': 0.019736842105263157, 'human': 0.019736842105263157, 'autom': 0.006578947368421052, 'code': 0.03289473684210526, 'review': 0.039473684210526314, 'mountain': 0.006578947368421052}


### 🔹 Log-Frequency Weighting

Log-frequency weighting adjusts term frequency to reduce the impact of very frequent terms.

Instead of raw TF, we use:

log-TF = 1 + log(tf)

This helps:

- Prevent very frequent terms from dominating
- Smooth large differences in term frequency

It is commonly used in vector space models for better performance.

In [8]:
# Log-frequency weighting

import math

def compute_log_tf(doc_term_counts):
    log_tf = []

    for counts in doc_term_counts:
        doc_log_tf = {}
        for term, count in counts.items():
            doc_log_tf[term] = 1 + math.log(count) if count > 0 else 0
        log_tf.append(doc_log_tf)

    return log_tf

log_tf = compute_log_tf(doc_term_counts)
print(dict(list(log_tf[0].items())[:10]))

{'real': 1.0, 'python': 3.302585092994046, 'podcast': 1.0, 'episod': 2.09861228866811, 'limit': 2.09861228866811, 'human': 2.09861228866811, 'autom': 1.0, 'code': 2.6094379124341005, 'review': 2.791759469228055, 'mountain': 1.0}


### 🔹 Document Frequency (DF)

Document Frequency measures how many documents contain a given term.

DF(t) = number of documents containing term t

Terms that appear in many documents are less useful for distinguishing between documents.

DF is used to compute IDF.

In [9]:
# Document frequency

from collections import defaultdict

def compute_df(doc_term_counts):
    df = defaultdict(int)

    for counts in doc_term_counts:
        for term in counts.keys():
            df[term] += 1

    return df

df = compute_df(doc_term_counts)
print(dict(list(df.items())[:10]))

{'real': 40, 'python': 40, 'podcast': 13, 'episod': 11, 'limit': 15, 'human': 2, 'autom': 6, 'code': 27, 'review': 15, 'mountain': 2}


### 🔹 Inverse Document Frequency (IDF)

Inverse Document Frequency measures how important a term is across the entire collection.

IDF(t) = log(N / DF(t))

Where:
- N = total number of documents
- DF(t) = document frequency of term t

Rare terms get higher IDF values, while common terms get lower values.

This helps emphasize discriminative words.

In [10]:
# Inverse document frequency

def compute_idf(df, total_docs):
    idf = {}

    for term, freq in df.items():
        idf[term] = math.log(total_docs / freq)

    return idf

idf = compute_idf(df, len(documents))
print(dict(list(idf.items())[:10]))

{'real': 0.0, 'python': 0.0, 'podcast': 1.1239300966523995, 'episod': 1.2909841813155658, 'limit': 0.9808292530117262, 'human': 2.995732273553991, 'autom': 1.8971199848858813, 'code': 0.3930425881096072, 'review': 0.9808292530117262, 'mountain': 2.995732273553991}


### 🔹 TF-IDF Weighting

TF-IDF combines term frequency and inverse document frequency:

TF-IDF(t, d) = TF(t, d) × IDF(t)

It reflects:

- High value → term is frequent in a document but rare overall
- Low value → term is common across many documents

TF-IDF is one of the most widely used weighting schemes in Information Retrieval.

In [11]:
# TF-IDF weighting

def compute_tfidf(tf, idf):
    tfidf = []

    for doc_tf in tf:
        doc_tfidf = {}
        for term, tf_val in doc_tf.items():
            doc_tfidf[term] = tf_val * idf.get(term, 0)
        tfidf.append(doc_tfidf)

    return tfidf

tfidf = compute_tfidf(tf, idf)
print(dict(list(tfidf[0].items())[:10]))

{'real': 0.0, 'python': 0.0, 'podcast': 0.007394276951660523, 'episod': 0.025479950947017743, 'limit': 0.019358472098915648, 'human': 0.05912629487277613, 'autom': 0.012481052532143955, 'code': 0.0129290325036055, 'review': 0.038716944197831296, 'mountain': 0.019708764957592044}


### 🔹 Cosine Similarity Retrieval

Cosine similarity measures the similarity between two vectors (e.g., query and document).

It is defined as:

cosine_similarity = (A · B) / (||A|| × ||B||)

Where:
- A = query vector
- B = document vector

The result ranges from:
- 1 → very similar
- 0 → no similarity

In this project, cosine similarity is used to rank documents based on their relevance to a query.

Documents with higher similarity scores are considered more relevant.

In [12]:
# Cosine similarity retrieval

def build_vocabulary(doc_term_counts):
    vocab = sorted(set(term for doc in doc_term_counts for term in doc.keys()))
    return vocab

def dict_to_vector(term_weights, vocab):
    return [term_weights.get(term, 0.0) for term in vocab]

def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    norm1 = math.sqrt(sum(a * a for a in vec1))
    norm2 = math.sqrt(sum(b * b for b in vec2))

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)

def build_query_tfidf(query, idf):
    query_tokens = preprocess_text(query)
    query_counts = Counter(query_tokens)

    total_terms = sum(query_counts.values())
    query_tf = {}

    for term, count in query_counts.items():
        query_tf[term] = count / total_terms

    query_tfidf = {}
    for term, tf_val in query_tf.items():
        query_tfidf[term] = tf_val * idf.get(term, 0.0)

    return query_tfidf

def retrieve_documents(query, tfidf, doc_term_counts, idf, documents, top_k=5):
    vocab = build_vocabulary(doc_term_counts)

    query_tfidf = build_query_tfidf(query, idf)
    query_vector = dict_to_vector(query_tfidf, vocab)

    results = []

    for doc_id, doc_tfidf in enumerate(tfidf):
        doc_vector = dict_to_vector(doc_tfidf, vocab)
        score = cosine_similarity(query_vector, doc_vector)
        results.append((doc_id, score))

    results = sorted(results, key=lambda x: x[1], reverse=True)

    print(f"Query: {query}\n")
    for rank, (doc_id, score) in enumerate(results[:top_k], start=1):
        print(f"Rank {rank}: Doc_{doc_id} | Score = {score:.4f}")
        print(documents[doc_id][:200].replace("\n", " "), "\n")

    return results[:top_k]

query = "machine learning data science"
top_results = retrieve_documents(query, tfidf, doc_term_counts, idf, documents, top_k=5)

Query: machine learning data science

Rank 1: Doc_8 | Score = 0.1450
Spyder: Your IDE for Data Science Development in Python  — FREE Email Series — 🐍 Python Tricks 💌  🔒 No spam. Unsubscribe any time. Table of Contents Table of Contents There are many different integrat 

Rank 2: Doc_0 | Score = 0.0684
The Real Python Podcast – Episode #289: Limitations in Human and Automated Code Review  With the mountains of Python code that it’s possible to generate now, how’s your code review going? What are the 

Rank 3: Doc_35 | Score = 0.0555
Quiz: Using Data Classes in Python  Interactive Quiz ⋅9QuestionsByJoseph Peart Revisit how Python data classes work withPython Data Classes. You’ll review how to define data classes, add default value 

Rank 4: Doc_17 | Score = 0.0369
Automate Python Data Analysis With YData Profiling  — FREE Email Series — 🐍 Python Tricks 💌  🔒 No spam. Unsubscribe any time. Table of Contents Table of Contents The YData Profiling package generates  

Rank 5: Doc_36 | Score =

### Part C — Querying

Create at least **5 information needs** and convert each into one or more queries.

For each query:

- retrieve the top documents
- explain why the retrieved documents are or are not relevant
- compare retrieval using at least **two different representations**, such as:
  - binary incidence
  - raw TF
  - TF-IDF

## Information Need 1: Python documentation and code readability

### Query 1
**"python docstrings documentation"**

**Relevant Documents**
- blog_31 (Highly Relevant)  
- blog_19 (Partially Relevant)  

### Query 2
**"write python code documentation"**

**Relevant Documents**
- blog_31 (Highly Relevant)  
- blog_19 (Partially Relevant)  

### Explanation
blog_31 is highly relevant because it directly explains how to write Python docstrings and documentation best practices.  
blog_19 is partially relevant as it discusses tutorial writing and content quality, which relates to documentation but not specifically to docstrings.

### Representation Comparison
- **Binary**: retrieves many documents containing "python" regardless of relevance  
- **TF**: improves ranking slightly by considering term frequency  
- **TF-IDF**: performs best because "docstrings" is a rare and meaningful term  

---

## Information Need 2: Python testing with mock objects

### Query 1
**"python mock testing"**

**Relevant Documents**
- blog_19 (Highly Relevant)  
- blog_12 (Highly Relevant)  

### Query 2
**"python mock object library"**

**Relevant Documents**
- blog_19 (Highly Relevant)  
- blog_12 (Highly Relevant)  

### Explanation
blog_19 is highly relevant because it focuses on improving tests using Python’s mock object library.  
blog_12 is also highly relevant since it discusses mocking and testing challenges in Python.

### Representation Comparison
- **Binary**: retrieves many irrelevant documents  
- **TF**: improves ranking but still favors common terms  
- **TF-IDF**: performs best due to emphasis on rare terms like "mock"  

---

## Information Need 3: Python with AI / LLM

### Query 1
**"python local llm ollama"**

**Relevant Documents**
- blog_34 (Highly Relevant)  

### Query 2
**"use python with ai models"**

**Relevant Documents**
- blog_34 (Highly Relevant)  

### Explanation
blog_34 is highly relevant because it discusses running local LLMs and integrating them with Python.

### Representation Comparison
- **Binary**: retrieves broad results due to common terms  
- **TF**: provides moderate improvement  
- **TF-IDF**: performs best by highlighting rare terms like "LLM"  

---

## Information Need 4: Python community and podcast updates

### Query 1
**"python community podcast updates"**

**Relevant Documents**
- blog_26 (Highly Relevant)  
- blog_34 (Highly Relevant)  

### Query 2
**"python developer community news"**

**Relevant Documents**
- blog_26 (Highly Relevant)  
- blog_34 (Highly Relevant)  

### Explanation
blog_26 and blog_34 are relevant because they discuss Python community topics and podcast-style updates.

### Representation Comparison
- **Binary**: retrieves many unrelated documents  
- **TF**: improves ranking slightly  
- **TF-IDF**: performs best due to distinguishing terms like "podcast" and "community"  

---

## Information Need 5: Python tools and development workflow

### Query 1
**"python development tools workflow"**

**Relevant Documents**
- blog_34 (Highly Relevant)  
- blog_11 (Partially Relevant)  

### Query 2
**"improve python development productivity"**

**Relevant Documents**
- blog_34 (Highly Relevant)  
- blog_11 (Partially Relevant)  

### Explanation
blog_34 is highly relevant because it discusses tools and development workflows.  
blog_11 is partially relevant as it relates to Python development practices but not specific tools.

### Representation Comparison
- **Binary**: retrieves too many general documents  
- **TF**: improves ranking slightly  
- **TF-IDF**: performs best due to focus on specific technical terms  

---

## Final Conclusion

TF-IDF consistently provides the most accurate results because:

- It reduces the influence of common terms (e.g., "python")  
- It emphasizes rare and meaningful keywords (e.g., "docstrings", "mock", "LLM")  
- It improves ranking consistency across different queries  

In [13]:
# Querying

import math
from collections import Counter
import pandas as pd

# --------------------------------------------------
# 1. Helper functions
# --------------------------------------------------

def build_vocabulary(doc_term_counts):
    vocab = sorted(set(term for doc in doc_term_counts for term in doc.keys()))
    return vocab

def dict_to_vector(term_weights, vocab):
    return [term_weights.get(term, 0.0) for term in vocab]

def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    norm1 = math.sqrt(sum(a * a for a in vec1))
    norm2 = math.sqrt(sum(b * b for b in vec2))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

def build_binary_doc_vectors(doc_term_counts, vocab):
    vectors = []
    for counts in doc_term_counts:
        vec = [1 if term in counts else 0 for term in vocab]
        vectors.append(vec)
    return vectors

def build_tf_doc_vectors(tf, vocab):
    return [dict_to_vector(doc_tf, vocab) for doc_tf in tf]

def build_tfidf_doc_vectors(tfidf, vocab):
    return [dict_to_vector(doc_tfidf, vocab) for doc_tfidf in tfidf]

def build_query_binary(query, vocab):
    tokens = preprocess_text(query)
    token_set = set(tokens)
    return [1 if term in token_set else 0 for term in vocab]

def build_query_tf(query, vocab):
    tokens = preprocess_text(query)
    counts = Counter(tokens)
    total = sum(counts.values())
    if total == 0:
        return [0.0] * len(vocab)
    tf_query = {term: count / total for term, count in counts.items()}
    return dict_to_vector(tf_query, vocab)

def build_query_tfidf(query, vocab, idf):
    tokens = preprocess_text(query)
    counts = Counter(tokens)
    total = sum(counts.values())
    if total == 0:
        return [0.0] * len(vocab)
    tf_query = {term: count / total for term, count in counts.items()}
    tfidf_query = {term: tf_query[term] * idf.get(term, 0.0) for term in tf_query}
    return dict_to_vector(tfidf_query, vocab)

def rank_documents(query_vector, doc_vectors, filenames, top_k=5):
    scores = []
    for i, doc_vector in enumerate(doc_vectors):
        score = cosine_similarity(query_vector, doc_vector)
        scores.append((filenames[i], score))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores[:top_k]

def relevance_label(doc_name, relevant_set):
    if doc_name in relevant_set:
        return "Relevant"
    return "Not Relevant"

# --------------------------------------------------
# 2. Make sure filenames, documents, tf, tfidf, idf exist
# --------------------------------------------------
# Required variables from earlier steps:
# filenames
# documents
# doc_term_counts
# tf
# tfidf
# idf

vocab = build_vocabulary(doc_term_counts)

binary_doc_vectors = build_binary_doc_vectors(doc_term_counts, vocab)
tf_doc_vectors = build_tf_doc_vectors(tf, vocab)
tfidf_doc_vectors = build_tfidf_doc_vectors(tfidf, vocab)

# --------------------------------------------------
# 3. Information needs + queries + manual relevance judgments
# --------------------------------------------------
# You can edit these if your interpretation changes.

part_c_tasks = {
    "Information Need 1: Python documentation and code readability": {
        "queries": [
            "python docstrings documentation",
            "write python code documentation"
        ],
        "relevant_docs": {"blog_31.txt", "blog_19.txt"}
    },
    "Information Need 2: Python testing with mock objects": {
        "queries": [
            "python mock testing",
            "python mock object library"
        ],
        "relevant_docs": {"blog_33.txt", "blog_26.txt"}
    },
    "Information Need 3: Python with AI / LLM": {
        "queries": [
            "python local llm ollama",
            "use python with ai models"
        ],
        "relevant_docs": {"blog_35.txt", "blog_34.txt"}
    },
    "Information Need 4: Python community and podcast updates": {
        "queries": [
            "python community podcast updates",
            "python developer community news"
        ],
        "relevant_docs": {"blog_26.txt", "blog_34.txt"}
    },
    "Information Need 5: Python tools and development workflow": {
        "queries": [
            "python development tools workflow",
            "improve python development productivity"
        ],
        "relevant_docs": {"blog_35.txt", "blog_30.txt", "blog_34.txt"}
    }
}

# --------------------------------------------------
# 4. Run Part C retrieval
# --------------------------------------------------

all_results = []

for info_need, config in part_c_tasks.items():
    print("=" * 100)
    print(info_need)
    print("Relevant Docs:", config["relevant_docs"])
    print()

    for query in config["queries"]:
        print("-" * 100)
        print("Query:", query)

        # Binary
        q_binary = build_query_binary(query, vocab)
        binary_results = rank_documents(q_binary, binary_doc_vectors, filenames, top_k=5)

        # Raw TF
        q_tf = build_query_tf(query, vocab)
        tf_results = rank_documents(q_tf, tf_doc_vectors, filenames, top_k=5)

        # TF-IDF
        q_tfidf = build_query_tfidf(query, vocab, idf)
        tfidf_results = rank_documents(q_tfidf, tfidf_doc_vectors, filenames, top_k=5)

        print("\nBinary Incidence Top 5:")
        for rank, (doc_name, score) in enumerate(binary_results, start=1):
            label = relevance_label(doc_name, config["relevant_docs"])
            print(f"{rank}. {doc_name} | score={score:.4f} | {label}")

        print("\nRaw TF Top 5:")
        for rank, (doc_name, score) in enumerate(tf_results, start=1):
            label = relevance_label(doc_name, config["relevant_docs"])
            print(f"{rank}. {doc_name} | score={score:.4f} | {label}")

        print("\nTF-IDF Top 5:")
        for rank, (doc_name, score) in enumerate(tfidf_results, start=1):
            label = relevance_label(doc_name, config["relevant_docs"])
            print(f"{rank}. {doc_name} | score={score:.4f} | {label}")

        # Save results in table form
        for method, results in [
            ("Binary", binary_results),
            ("Raw TF", tf_results),
            ("TF-IDF", tfidf_results)
        ]:
            for rank, (doc_name, score) in enumerate(results, start=1):
                all_results.append({
                    "Information Need": info_need,
                    "Query": query,
                    "Method": method,
                    "Rank": rank,
                    "Document": doc_name,
                    "Score": round(score, 4),
                    "Relevant?": relevance_label(doc_name, config["relevant_docs"])
                })

        print()

# --------------------------------------------------
# 5. Save results to DataFrame
# --------------------------------------------------

part_c_results_df = pd.DataFrame(all_results)
part_c_results_df.head(20)

Information Need 1: Python documentation and code readability
Relevant Docs: {'blog_31.txt', 'blog_19.txt'}

----------------------------------------------------------------------------------------------------
Query: python docstrings documentation

Binary Incidence Top 5:
1. blog_31.txt | score=0.1901 | Relevant
2. blog_35.txt | score=0.1543 | Not Relevant
3. blog_33.txt | score=0.1275 | Not Relevant
4. blog_19.txt | score=0.1210 | Relevant
5. blog_28.txt | score=0.1185 | Not Relevant

Raw TF Top 5:
1. blog_38.txt | score=0.4176 | Not Relevant
2. blog_22.txt | score=0.3740 | Not Relevant
3. blog_31.txt | score=0.3722 | Relevant
4. blog_28.txt | score=0.3644 | Not Relevant
5. blog_19.txt | score=0.3397 | Relevant

TF-IDF Top 5:
1. blog_31.txt | score=0.2180 | Relevant
2. blog_35.txt | score=0.0788 | Not Relevant
3. blog_33.txt | score=0.0714 | Not Relevant
4. blog_19.txt | score=0.0632 | Relevant
5. blog_37.txt | score=0.0626 | Not Relevant

--------------------------------------------

,Information Need,Query,Method,Rank,Document,Score,Relevant?
0,Information Need 1: Python documentation and c...,python docstrings documentation,Binary,1,blog_31.txt,0.1901,Relevant
1,Information Need 1: Python documentation and c...,python docstrings documentation,Binary,2,blog_35.txt,0.1543,Not Relevant
2,Information Need 1: Python documentation and c...,python docstrings documentation,Binary,3,blog_33.txt,0.1275,Not Relevant
3,Information Need 1: Python documentation and c...,python docstrings documentation,Binary,4,blog_19.txt,0.1210,Relevant
4,Information Need 1: Python documentation and c...,python docstrings documentation,Binary,5,blog_28.txt,0.1185,Not Relevant
5,Information Need 1: Python documentation and c...,python docstrings documentation,Raw TF,1,blog_38.txt,0.4176,Not Relevant
6,Information Need 1: Python documentation and c...,python docstrings documentation,Raw TF,2,blog_22.txt,0.3740,Not Relevant
7,Information Need 1: Python documentation and c...,python docstrings documentation,Raw TF,3,blog_31.txt,0.3722,Relevant
8,Information Need 1: Python documentation and c...,python docstrings documentation,Raw TF,4,blog_28.txt,0.3644,Not Relevant
9,Information Need 1: Python documentation and c...,python docstrings documentation,Raw TF,5,blog_19.txt,0.3397,Relevant


## Analysis for Query: "python docstrings documentation"

### Observations

Across the three representations (Binary, Raw TF, TF-IDF), we observe different ranking behaviors:

- **Binary Incidence**
  - Correctly retrieves **blog_31.txt** as Rank 1 (Relevant)
  - Also retrieves **blog_19.txt** (Relevant) at Rank 4
  - However, several non-relevant documents appear in top ranks

- **Raw TF**
  - Ranks **non-relevant documents (blog_38, blog_22)** at the top
  - Relevant document **blog_31.txt** appears only at Rank 3
  - Shows sensitivity to term frequency rather than importance

- **TF-IDF**
  - Correctly ranks **blog_31.txt** as Rank 1 (Relevant)
  - Also retrieves **blog_19.txt** within Top 5
  - Non-relevant documents have lower scores compared to TF

---

### Interpretation

- **Binary model**
  - Works reasonably well for simple queries
  - But lacks weighting, so irrelevant documents with matching terms are ranked highly

- **Raw TF model**
  - Overemphasizes frequent words such as "python"
  - Leads to irrelevant documents dominating top ranks

- **TF-IDF model**
  - Performs best overall
  - Successfully prioritizes **"docstrings"**, which is a rare and meaningful term
  - Produces more accurate and stable rankings

---

### Conclusion

TF-IDF provides the most effective retrieval for this query because it balances:

- term importance (TF)
- term rarity (IDF)

This allows it to distinguish highly relevant documents (like blog_31.txt) from less relevant ones more effectively than Binary and Raw TF.

### Part D — Evaluation

For at least **3 queries**, create relevance judgments and compute:

- confusion matrix
- precision
- recall
- F1-score
- Precision@K
- Average Precision (AP)
- Mean Reciprocal Rank (MRR)


# Part D — Evaluation Metrics

To evaluate the effectiveness of the information retrieval system, several standard evaluation metrics are used. These metrics help measure how well the system retrieves relevant documents and ranks them appropriately.

---

## 1. Confusion Matrix

The confusion matrix summarizes retrieval results using four components:

- **True Positive (TP):** Relevant documents correctly retrieved  
- **False Positive (FP):** Non-relevant documents retrieved  
- **False Negative (FN):** Relevant documents not retrieved  
- **True Negative (TN):** Non-relevant documents not retrieved  

In information retrieval, TN is usually not emphasized because the number of non-relevant documents is very large.

---

## 2. Precision

Precision measures the proportion of retrieved documents that are relevant:

$$
Precision = \frac{TP}{TP + FP}
$$

- High precision means fewer irrelevant documents are retrieved.

---

## 3. Recall

Recall measures the proportion of relevant documents that are successfully retrieved:

$$
Recall = \frac{TP}{TP + FN}
$$

- High recall means most relevant documents are found.

---

## 4. F1-score

F1-score is the harmonic mean of precision and recall:

$$
F1 = \frac{2 \times Precision \times Recall}{Precision + Recall}
$$

- It balances precision and recall into a single metric.

---

## 5. Precision@K

Precision@K evaluates precision at the top **K** retrieved documents:

$$
Precision@K = \frac{\text{Number of relevant documents in top K}}{K}
$$

- Focuses on the quality of top-ranked results.
- Important because users usually only look at the first few results.

---

## 6. Average Precision (AP)

Average Precision considers both precision and ranking order:

$$
AP = \frac{1}{|R|} \sum_{k=1}^{N} Precision@k \cdot rel(k)
$$

Where:
- $ |R| $ = number of relevant documents  
- $ rel(k) $ = 1 if the document at rank k is relevant, otherwise 0  

- Rewards systems that rank relevant documents higher.

---

## 7. Mean Reciprocal Rank (MRR)

MRR evaluates how quickly the first relevant document appears:

$$
MRR = \frac{1}{Q} \sum_{i=1}^{Q} \frac{1}{rank_i}
$$

Where:
- $ Q $ = number of queries  
- $ rank_i $ = position of the first relevant document for query i  

- Higher MRR means relevant results appear earlier in rankings.

---

## Summary

These metrics together provide a comprehensive evaluation:

- **Precision / Recall / F1** → overall retrieval quality  
- **Precision@K** → top results quality  
- **AP** → ranking quality  
- **MRR** → speed of finding first relevant result  

Using multiple metrics ensures a more complete and reliable evaluation of the retrieval system.

In [14]:
# Part D — Evaluation

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Evaluation helper functions
# --------------------------------------------------

def confusion_matrix_at_k(results, relevant_set, k=5):
    tp = fp = fn = tn = 0

    retrieved_docs = [doc for doc, _ in results[:k]]

    for doc in retrieved_docs:
        if doc in relevant_set:
            tp += 1
        else:
            fp += 1

    for doc in relevant_set:
        if doc not in retrieved_docs:
            fn += 1

    # TN is not very meaningful in IR (huge corpus), keep as 0
    return tp, fp, fn, tn


def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0


def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0


def f1(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0


def precision_at_k(results, relevant_set, k=5):
    retrieved_docs = [doc for doc, _ in results[:k]]
    relevant_retrieved = sum(1 for doc in retrieved_docs if doc in relevant_set)
    return relevant_retrieved / k


def average_precision(results, relevant_set):
    score = 0.0
    relevant_count = 0

    for i, (doc, _) in enumerate(results, start=1):
        if doc in relevant_set:
            relevant_count += 1
            score += relevant_count / i

    if len(relevant_set) == 0:
        return 0

    return score / len(relevant_set)


def reciprocal_rank(results, relevant_set):
    for i, (doc, _) in enumerate(results, start=1):
        if doc in relevant_set:
            return 1 / i
    return 0


# --------------------------------------------------
# 2. Select at least 3 queries
# --------------------------------------------------

evaluation_queries = [
    ("python docstrings documentation", {"blog_31.txt", "blog_19.txt"}),
    ("python mock object library", {"blog_33.txt", "blog_26.txt"}),
    ("python local llm ollama", {"blog_34.txt"})
]

methods = {
    "Binary": binary_doc_vectors,
    "Raw TF": tf_doc_vectors,
    "TF-IDF": tfidf_doc_vectors
}

query_builders = {
    "Binary": lambda q: build_query_binary(q, vocab),
    "Raw TF": lambda q: build_query_tf(q, vocab),
    "TF-IDF": lambda q: build_query_tfidf(q, vocab, idf)
}

# --------------------------------------------------
# 3. Run evaluation
# --------------------------------------------------

results_list = []

for query, relevant_set in evaluation_queries:
    print("=" * 100)
    print("Query:", query)
    print("Relevant Docs:", relevant_set)

    for method in methods:
        q_vec = query_builders[method](query)
        ranked = rank_documents(q_vec, methods[method], filenames, top_k=10)

        tp, fp, fn, tn = confusion_matrix_at_k(ranked, relevant_set, k=5)

        p = precision(tp, fp)
        r = recall(tp, fn)
        f1_score = f1(p, r)
        p_at_k = precision_at_k(ranked, relevant_set, k=5)
        ap = average_precision(ranked, relevant_set)
        rr = reciprocal_rank(ranked, relevant_set)

        print(f"\nMethod: {method}")
        print(f"TP={tp}, FP={fp}, FN={fn}")
        print(f"Precision={p:.3f}, Recall={r:.3f}, F1={f1_score:.3f}")
        print(f"P@5={p_at_k:.3f}, AP={ap:.3f}, RR={rr:.3f}")

        results_list.append({
            "Query": query,
            "Method": method,
            "Precision": p,
            "Recall": r,
            "F1": f1_score,
            "P@5": p_at_k,
            "AP": ap,
            "RR": rr
        })

# --------------------------------------------------
# 4. Compute MRR
# --------------------------------------------------

df_eval = pd.DataFrame(results_list)

mrr = df_eval.groupby("Method")["RR"].mean()

print("\n" + "=" * 100)
print("Mean Reciprocal Rank (MRR):")
print(mrr)

df_eval

Query: python docstrings documentation
Relevant Docs: {'blog_31.txt', 'blog_19.txt'}

Method: Binary
TP=2, FP=3, FN=0
Precision=0.400, Recall=1.000, F1=0.571
P@5=0.400, AP=0.750, RR=1.000

Method: Raw TF
TP=2, FP=3, FN=0
Precision=0.400, Recall=1.000, F1=0.571
P@5=0.400, AP=0.367, RR=0.333

Method: TF-IDF
TP=2, FP=3, FN=0
Precision=0.400, Recall=1.000, F1=0.571
P@5=0.400, AP=0.750, RR=1.000
Query: python mock object library
Relevant Docs: {'blog_33.txt', 'blog_26.txt'}

Method: Binary
TP=2, FP=3, FN=0
Precision=0.400, Recall=1.000, F1=0.571
P@5=0.400, AP=0.833, RR=1.000

Method: Raw TF
TP=2, FP=3, FN=0
Precision=0.400, Recall=1.000, F1=0.571
P@5=0.400, AP=1.000, RR=1.000

Method: TF-IDF
TP=2, FP=3, FN=0
Precision=0.400, Recall=1.000, F1=0.571
P@5=0.400, AP=1.000, RR=1.000
Query: python local llm ollama
Relevant Docs: {'blog_34.txt'}

Method: Binary
TP=0, FP=5, FN=1
Precision=0.000, Recall=0.000, F1=0.000
P@5=0.000, AP=0.000, RR=0.000

Method: Raw TF
TP=0, FP=5, FN=1
Precision=0.000, Re

,Query,Method,Precision,Recall,F1,P@5,AP,RR
0,python docstrings documentation,Binary,0.4,1.0,0.571429,0.4,0.750000,1.000000
1,python docstrings documentation,Raw TF,0.4,1.0,0.571429,0.4,0.366667,0.333333
2,python docstrings documentation,TF-IDF,0.4,1.0,0.571429,0.4,0.750000,1.000000
3,python mock object library,Binary,0.4,1.0,0.571429,0.4,0.833333,1.000000
4,python mock object library,Raw TF,0.4,1.0,0.571429,0.4,1.000000,1.000000
5,python mock object library,TF-IDF,0.4,1.0,0.571429,0.4,1.000000,1.000000
6,python local llm ollama,Binary,0.0,0.0,0.000000,0.0,0.000000,0.000000
7,python local llm ollama,Raw TF,0.0,0.0,0.000000,0.0,0.000000,0.000000
8,python local llm ollama,TF-IDF,0.0,0.0,0.000000,0.0,0.000000,0.000000


## Part D — Evaluation Results

### Overview

Three queries were evaluated using Binary, Raw TF, and TF-IDF representations.  
The evaluation includes Precision, Recall, F1-score, Precision@5, Average Precision (AP), and Mean Reciprocal Rank (MRR).

---

## Results Analysis

### Query 1: python docstrings documentation

- All three models achieved:
  - Precision = 0.40
  - Recall = 1.00
  - F1-score = 0.571

- However, differences appear in ranking quality:
  - **TF-IDF and Binary achieved higher AP (0.750)** compared to Raw TF (0.367)
  - **RR = 1.0 for Binary and TF-IDF**, meaning the first relevant document was ranked first
  - Raw TF has lower RR (0.333), indicating delayed retrieval of relevant documents

👉 Interpretation:
- TF-IDF and Binary perform better in ranking relevant documents earlier
- Raw TF struggles due to overemphasis on frequent terms

---

### Query 2: python mock object library

- All models retrieved both relevant documents:
  - Precision = 0.40
  - Recall = 1.00

- Ranking quality:
  - **Raw TF and TF-IDF achieved perfect AP (1.000)**
  - Binary achieved slightly lower AP (0.833)
  - All methods achieved RR = 1.0

👉 Interpretation:
- This query contains more distinctive keywords ("mock", "object")
- All models perform well, especially TF-IDF and Raw TF
- TF-IDF maintains strong ranking consistency

---

### Query 3: python local llm ollama

- All models failed completely:
  - Precision = 0.00
  - Recall = 0.00
  - F1 = 0.00
  - AP = 0.00
  - RR = 0.00

👉 Interpretation:
- None of the models retrieved the relevant document (blog_34.txt) in top 5
- Possible reasons:
  - Rare or unseen terms ("ollama", "llm")
  - Vocabulary mismatch between query and documents
  - Insufficient term overlap

👉 This highlights a limitation of:
- Bag-of-words models
- TF-IDF when query terms do not match document vocabulary well

---

## Mean Reciprocal Rank (MRR)

| Method   | MRR   |
|----------|------|
| Binary   | 0.667 |
| Raw TF   | 0.444 |
| TF-IDF   | 0.667 |

### Interpretation

- **TF-IDF and Binary achieve the highest MRR (0.667)**
- **Raw TF performs worse (0.444)** due to poorer ranking of relevant documents
- Higher MRR indicates that relevant documents are retrieved earlier

---

## Key Insights

- **TF-IDF performs best overall**
  - Strong ranking performance (high AP and MRR)
  - Consistent across different queries

- **Raw TF is unstable**
  - Performs well for some queries
  - Fails in ranking consistency

- **Binary model is surprisingly competitive**
  - Performs well when relevant terms exist
  - But lacks weighting and ranking refinement

- **Major limitation observed**
  - All models fail when query terms are rare or mismatched (Query 3)
  - Indicates need for:
    - synonym handling
    - semantic search
    - embeddings-based retrieval

---

## Conclusion

TF-IDF provides the most reliable performance across queries, especially in ranking relevant documents earlier.

However, traditional vector space models (Binary, TF, TF-IDF) struggle when:
- query terms are rare
- vocabulary mismatch occurs

This suggests that more advanced retrieval methods (e.g., semantic search) could further improve performance.

## Thinking Points 

**What is the main difference between Binary, TF and TF-IDF?**  
Binary only check if word exist or not, it does not care how many times. TF count frequency, so more times means higher score. TF-IDF is better because it also consider how rare the word is in all documents.

---

**Why TF-IDF perform better than other methods?**  
TF-IDF perform better because it reduce the weight of common words like "python". It give more importance to special words like "docstrings", so it can rank documents more correctly.

---

**Why Raw TF sometimes gives bad results?**  
Raw TF focus too much on frequency. If a document has many common words, it will get high score even if it is not really relevant. So it can bring wrong documents to top.

---

**What is the limitation of bag-of-words model?**  
Bag-of-words only look at words, it cannot understand meaning. So if two words have same meaning but different spelling, it cannot connect them.

---

**Why Precision is low but Recall is high in our results?**  
Recall is high because relevant documents are found. But precision is low because many non-relevant documents are also retrieved in top results.

---

**Why Precision@K and MRR are important?**  
Users usually only look at first few results. So if relevant document appear early, it is more useful for users.

---

